### Stage 1 - Data Loading & Filtering

In [1]:
import pandas as pd

print("Loading... please wait")
df = pd.read_csv('online_retail.csv', encoding='utf-8', low_memory=False)
print(f"Raw shape: {df.shape}")

Loading... please wait
Raw shape: (1048575, 8)


### Stage 2 - Data Cleaning

In [3]:
# Fix date parsing with dayfirst=True
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True)

# Filter Year 2010-2011 only
df = df[(df['InvoiceDate'].dt.year >= 2010) & (df['InvoiceDate'].dt.year <= 2011)]

# Filter UK only
df = df[df['Country'] == 'United Kingdom']

# Remove missing Customer ID
df = df.dropna(subset=['Customer ID'])

# Remove cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove bad quantities and prices
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

print(f"Filtered shape: {df.shape}")
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"\nUnique customers: {df['Customer ID'].nunique()}")
print(f"Unique invoices: {df['Invoice'].nunique()}")

Filtered shape: (685639, 8)

Date range: 2010-01-04 09:24:00 to 2011-12-04 13:15:00

Unique customers: 5252
Unique invoices: 31656


### Stage 3 - Demographic Enrichment

In [4]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Get unique customers
customers = df['Customer ID'].unique()

# Generate synthetic demographics per customer
customer_demographics = pd.DataFrame({
    'Customer ID': customers,
    'Gender': np.random.choice(['Male', 'Female'], size=len(customers), p=[0.45, 0.55]),
    'Age': np.random.randint(18, 70, size=len(customers))
})

# Create Age Band
def age_band(age):
    if age < 26:
        return '18-25'
    elif age < 36:
        return '26-35'
    elif age < 46:
        return '36-45'
    elif age < 56:
        return '46-55'
    else:
        return '56+'

customer_demographics['Age_Band'] = customer_demographics['Age'].apply(age_band)

# Merge back into main dataframe
df = df.merge(customer_demographics, on='Customer ID', how='left')

print(f"Shape after enrichment: {df.shape}")
print(f"\nGender distribution:\n{df['Gender'].value_counts()}")
print(f"\nAge Band distribution:\n{df['Age_Band'].value_counts()}")
print(f"\nColumns now: {list(df.columns)}")

Shape after enrichment: (685639, 11)

Gender distribution:
Gender
Female    369094
Male      316545
Name: count, dtype: int64

Age Band distribution:
Age_Band
56+      188572
26-35    146499
36-45    129517
46-55    119197
18-25    101854
Name: count, dtype: int64

Columns now: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Gender', 'Age', 'Age_Band']


In [5]:
# Add Revenue column (Quantity x Price)
df['Revenue'] = df['Quantity'] * df['Price']

# Save cleaned enriched file
df.to_csv('CustomerLens_Cleaned.csv', index=False)

print(f"Final shape: {df.shape}")
print(f"\nSample revenue stats:")
print(df['Revenue'].describe())
print(f"\nFile saved as CustomerLens_Cleaned.csv")

Final shape: (685639, 12)

Sample revenue stats:
count    685639.000000
mean         20.025205
std         127.392320
min           0.001000
25%           4.200000
50%          10.200000
75%          17.700000
max       77183.600000
Name: Revenue, dtype: float64

File saved as CustomerLens_Cleaned.csv


### Stage 4 — Feature Engineering

In [6]:
# Cohort Month + Cohort Index

# Extract invoice month
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Cohort Month :— first purchase month per customer
df['Cohort_Month'] = df.groupby('Customer ID')['InvoiceDate'].transform('min').dt.to_period('M')

# Cohort Index :— months since first purchase
df['Cohort_Index'] = (df['InvoiceMonth'] - df['Cohort_Month']).apply(lambda x: x.n)

# Verify
print(df[['Customer ID', 'InvoiceDate', 'Cohort_Month', 'InvoiceMonth', 'Cohort_Index']].head(15))
print(f"\nMax Cohort Index: {df['Cohort_Index'].max()}")

    Customer ID         InvoiceDate Cohort_Month InvoiceMonth  Cohort_Index
0       12346.0 2010-01-04 09:24:00      2010-01      2010-01             0
1       12346.0 2010-01-04 09:53:00      2010-01      2010-01             0
2       14590.0 2010-01-04 10:28:00      2010-01      2010-01             0
3       14590.0 2010-01-04 10:28:00      2010-01      2010-01             0
4       14590.0 2010-01-04 10:28:00      2010-01      2010-01             0
5       14590.0 2010-01-04 10:28:00      2010-01      2010-01             0
6       14590.0 2010-01-04 10:28:00      2010-01      2010-01             0
7       14590.0 2010-01-04 10:28:00      2010-01      2010-01             0
8       13287.0 2010-01-04 10:43:00      2010-01      2010-01             0
9       13287.0 2010-01-04 10:43:00      2010-01      2010-01             0
10      13287.0 2010-01-04 10:43:00      2010-01      2010-01             0
11      13287.0 2010-01-04 10:43:00      2010-01      2010-01             0
12      1328

In [7]:
# RFM Features

# Reference date — day after last transaction
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Aggregate per customer
rfm = df.groupby('Customer ID').agg(
    Last_Purchase=('InvoiceDate', 'max'),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

# Recency — days since last purchase
rfm['Recency'] = (reference_date - rfm['Last_Purchase']).dt.days

# CLTV — simple version
rfm['CLTV'] = rfm['Monetary'] * rfm['Frequency']

# Churn Flag — no purchase in last 90 days
rfm['Churn_Flag'] = (rfm['Recency'] > 90).astype(int)

print(rfm.head(10))
print(f"\nShape: {rfm.shape}")
print(f"\nChurn rate: {rfm['Churn_Flag'].mean():.1%}")
print(f"\nRFM Stats:")
print(rfm[['Recency', 'Frequency', 'Monetary', 'CLTV']].describe())

   Customer ID       Last_Purchase  Frequency  Monetary  Recency         CLTV  \
0      12346.0 2011-01-18 10:01:00          7  77442.96      321    542100.72   
1      12608.0 2010-10-31 10:49:00          1    415.79      400       415.79   
2      12745.0 2010-08-10 10:14:00          2    723.85      482      1447.70   
3      12746.0 2010-06-17 10:41:00          1    254.55      536       254.55   
4      12747.0 2011-11-17 17:13:00         22   7806.04       17    171732.88   
5      12748.0 2011-12-04 12:31:00        324  53992.72        1  17493641.28   
6      12749.0 2011-11-17 12:05:00          8   6134.30       18     49074.40   
7      12777.0 2010-09-08 11:35:00          1    519.45      453       519.45   
8      12819.0 2010-09-07 15:20:00          1    540.52      453       540.52   
9      12820.0 2011-10-26 13:27:00          9   2330.87       39     20977.83   

   Churn_Flag  
0           1  
1           1  
2           1  
3           1  
4           0  
5           

### Stage 5 — RFM Scoring & Segmentation

In [8]:
# RFM Scoring — quartile based
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4])

# Combined RFM Score
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Segment labels
def rfm_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])
    
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champion'
    elif r >= 3 and f >= 2:
        return 'Loyal'
    elif r >= 2 and f >= 2:
        return 'Potential Loyalist'
    elif r >= 3 and f <= 1:
        return 'New Customer'
    elif r == 2 and f <= 2:
        return 'At Risk'
    elif r == 1 and f >= 2:
        return 'Cannot Lose Them'
    else:
        return 'Lost'

rfm['RFM_Segment'] = rfm.apply(rfm_segment, axis=1)

print(rfm['RFM_Segment'].value_counts())
print(f"\nTotal customers: {rfm.shape[0]}")

RFM_Segment
Champion              1623
Potential Loyalist     996
Loyal                  697
Lost                   688
Cannot Lose Them       623
At Risk                315
New Customer           310
Name: count, dtype: int64

Total customers: 5252


In [9]:
# Merge Everything + Export

# Merge demographics into RFM table
customer_level = rfm.merge(customer_demographics, on='Customer ID', how='left')

# Merge cohort info into transaction level
df_final = df.merge(rfm[['Customer ID', 'Recency', 'Frequency', 
                           'Monetary', 'CLTV', 'Churn_Flag', 
                           'RFM_Segment', 'R_Score', 'F_Score', 'M_Score']], 
                    on='Customer ID', how='left')

# Export 1 — Customer level (for RFM, CLTV, Churn visuals)
customer_level.to_csv('CustomerLens_Customer_Level.csv', index=False)

# Export 2 — Transaction level (for Cohort visuals)
df_final.to_csv('CustomerLens_Transaction_Level.csv', index=False)

print(f"Customer level shape: {customer_level.shape}")
print(f"Transaction level shape: {df_final.shape}")
print(f"\nCustomer level columns: {list(customer_level.columns)}")
print(f"\nTransaction level columns: {list(df_final.columns)}")

Customer level shape: (5252, 15)
Transaction level shape: (685639, 24)

Customer level columns: ['Customer ID', 'Last_Purchase', 'Frequency', 'Monetary', 'Recency', 'CLTV', 'Churn_Flag', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Segment', 'Gender', 'Age', 'Age_Band']

Transaction level columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Gender', 'Age', 'Age_Band', 'Revenue', 'InvoiceMonth', 'Cohort_Month', 'Cohort_Index', 'Recency', 'Frequency', 'Monetary', 'CLTV', 'Churn_Flag', 'RFM_Segment', 'R_Score', 'F_Score', 'M_Score']
